# YOLOv11-seg & U-Net Training — Colab

## Setup options

**Option A (recommended):** Upload `dataset.zip`, `roi_data.zip`, and `unet_train.py` to your Google Drive, then mount Drive in Cell 2.

**Option B:** Upload the files directly to `/content/` using the Colab file browser (sidebar → 📁). `dataset.zip` is ~546 MB so Drive is more reliable.

After training, download `yolo_best.pt` and `unet_model.pth` and drop them in your local project root.

In [ ]:
# ═══ CONFIG — change epochs / model size here ═══

YOLO_MODEL  = "yolo11s-seg.pt"  # n=speed, s=balanced, m=accuracy, l=best
YOLO_EPOCHS = 50                 # was 10 locally
YOLO_IMGSZ  = 640
YOLO_BATCH  = 16                 # T4: 16 at 640; lower if OOM

UNET_EPOCHS = 100                # was 50 locally
UNET_LR     = 1e-4
UNET_BATCH  = 8
UNET_IMGSZ  = 256

# Set to True if uploading files to Drive instead of /content/
USE_DRIVE = True
DRIVE_FOLDER = "colab_train"  # folder name in your Drive

In [ ]:
# Install dependencies
!pip install ultralytics opencv-python-headless matplotlib tqdm pyyaml

In [ ]:
# Mount Google Drive (if USE_DRIVE=True) and copy files to /content/
import os, shutil, zipfile
from pathlib import Path

WORK_DIR = Path("/content")
os.chdir(WORK_DIR)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive") / DRIVE_FOLDER

    # Copy files from Drive to /content/ (faster than reading from Drive during training)
    for f in ["dataset.zip", "roi_data.zip", "unet_train.py"]:
        src = drive_dir / f
        dst = WORK_DIR / f
        if src.exists():
            print(f"  Copying {f} from Drive ...")
            shutil.copy2(src, dst)
        else:
            print(f"  ⚠ {f} not found in Drive folder")

# Verify files
print("\nFile check:")
for f in ["dataset.zip", "roi_data.zip", "unet_train.py"]:
    p = WORK_DIR / f
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        print(f"  ✓ {f}: {size_mb:.1f} MB")
    else:
        print(f"  ✗ {f}: MISSING")

if not (WORK_DIR / "dataset.zip").exists():
    raise FileNotFoundError("dataset.zip not found! Upload it to Drive or /content/")

In [ ]:
# Unzip datasets

# YOLO dataset
ds_zip = WORK_DIR / "dataset.zip"
if ds_zip.exists():
    print(f"Unzipping dataset.zip ({ds_zip.stat().st_size / 1e6:.0f} MB) ...")
    with zipfile.ZipFile(ds_zip, 'r') as z:
        z.extractall(WORK_DIR)
    print("✓ Extracted dataset.zip")
else:
    raise FileNotFoundError("dataset.zip not found!")

# ROI data for U-Net
roi_zip = WORK_DIR / "roi_data.zip"
if roi_zip.exists():
    print(f"Unzipping roi_data.zip ({roi_zip.stat().st_size / 1e6:.0f} MB) ...")
    with zipfile.ZipFile(roi_zip, 'r') as z:
        z.extractall(WORK_DIR)
    print("✓ Extracted roi_data.zip")
else:
    print("⚠ roi_data.zip not found — U-Net training will be skipped")

In [ ]:
# Fix data.yaml path for Colab
import yaml

data_yaml = WORK_DIR / "dataset" / "data.yaml"
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

cfg["path"] = str(WORK_DIR / "dataset")
with open(data_yaml, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"✓ data.yaml path set to: {cfg['path']}")

In [ ]:
# Train YOLOv11-seg
from ultralytics import YOLO
import shutil

model = YOLO(YOLO_MODEL)

results = model.train(
    data     = str(data_yaml),
    epochs   = YOLO_EPOCHS,
    imgsz    = YOLO_IMGSZ,
    batch    = YOLO_BATCH,
    device   = 0,
    project  = str(WORK_DIR / "runs" / "segment"),
    name     = "yolo11s_strawberry",
    exist_ok = True,
    verbose  = True,
)

best_src = WORK_DIR / "runs" / "segment" / "yolo11s_strawberry" / "weights" / "best.pt"
best_dst = WORK_DIR / "yolo_best.pt"
shutil.copy2(best_src, best_dst)
print(f"\n✅ YOLO training complete! Best weights → {best_dst}")

In [ ]:
# Print YOLO results summary
import csv

results_csv = WORK_DIR / "runs" / "segment" / "yolo11s_strawberry" / "results.csv"
if results_csv.exists():
    with open(results_csv) as f:
        reader = csv.DictReader(f)
        rows = [{k.strip(): float(v) for k, v in row.items() if v.strip()} for row in reader]

    last = rows[-1]
    best_mAP = max(rows, key=lambda r: r["metrics/mAP50(M)"])
    print("═══ YOLOv11-seg Results ═══")
    print(f"  Final epoch → mAP@0.5(box)={last['metrics/mAP50(B)']:.4f}  mAP@0.5(mask)={last['metrics/mAP50(M)']:.4f}")
    print(f"  Best epoch  → mAP@0.5(box)={best_mAP['metrics/mAP50(B)']:.4f}  mAP@0.5(mask)={best_mAP['metrics/mAP50(M)']:.4f}  (epoch {int(best_mAP['epoch'])})")
else:
    print("⚠ results.csv not found")

In [ ]:
# Train U-Net
ROI_DIR  = WORK_DIR / "roi_crops"
MASK_DIR = WORK_DIR / "roi_masks"

if ROI_DIR.exists() and MASK_DIR.exists() and (WORK_DIR / "unet_train.py").exists():
    %run unet_train.py
else:
    print("⚠ roi_crops/ or roi_masks/ or unet_train.py not found — skipping U-Net")

In [ ]:
# Download trained models
from google.colab import files

print("Downloading yolo_best.pt ...")
files.download(str(WORK_DIR / "yolo_best.pt"))

unet_path = WORK_DIR / "unet_model.pth"
if unet_path.exists():
    print("Downloading unet_model.pth ...")
    files.download(str(unet_path))